# Libraries required for tagging process 

In [16]:
# libraries required

import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
import pandas as pd
import matplotlib.pyplot as plt
import re

###  Data loading

In [17]:
data = pd.read_csv('bbc_news.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [18]:
titles = pd.DataFrame(data['title'])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


## Data Cleaning

#### Lower case

In [19]:
titles['Lowercase'] = titles['title'].str.lower()
titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      1000 non-null   object
 1   Lowercase  1000 non-null   object
dtypes: object(2)
memory usage: 15.8+ KB


#### Remove stopwords

In [21]:
en_stopwords = stopwords.words('english')
print(en_stopwords)


['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [22]:
titles['no_stopwords'] = titles['Lowercase'].apply(lambda x: ' '.join([word for word in x.split() if word not in en_stopwords]))
titles.head()

,title,Lowercase,no_stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds


#### Remove Punctuations


In [23]:
titles['no_punct'] = titles.apply(lambda x : re.sub(r'[^\w\s]', '', x['no_stopwords']), axis=1)
titles.head()

,title,Lowercase,no_stopwords,no_punct
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds


#### Tokenization

In [28]:
titles['Tokens_raw'] = titles.apply(lambda x: word_tokenize(x['title']), axis=1)
titles['Tokens_clean'] = titles.apply(lambda x: word_tokenize(x['no_punct']), axis=1)
titles.head()

,title,Lowercase,no_stopwords,no_punct,Tokens_clean,Lemmatized_clean,Tokens_raw
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]","[refuse, work]","[Can, I, refuse, to, work, ?]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco...","[platinum, jubilee, 70, year, queen, 70, second]","[Platinum, Jubilee, :, 70, years, of, the, Que..."


#### Lemmatization

In [29]:
lemmatizer = WordNetLemmatizer()
titles['Lemmatized_clean'] = titles['Tokens_clean'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
titles.head()

,title,Lowercase,no_stopwords,no_punct,Tokens_clean,Lemmatized_clean,Tokens_raw
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]","[refuse, work]","[Can, I, refuse, to, work, ?]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic...","[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com...","[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga...","[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco...","[platinum, jubilee, 70, year, queen, 70, second]","[Platinum, Jubilee, :, 70, years, of, the, Que..."


#### Tokens

In [31]:
tokens_raw_list = sum(titles['Tokens_raw'],[])
tokens_clean_list = sum(titles['Tokens_clean'],[])

print(tokens_raw_list)

['Can', 'I', 'refuse', 'to', 'work', '?', "'Liz", 'Truss', 'the', 'Brief', '?', "'", 'World', 'reacts', 'to', 'UK', 'political', 'turmoil', 'Rationing', 'energy', 'is', 'nothing', 'new', 'for', 'off-grid', 'community', 'The', 'hunt', 'for', 'superyachts', 'of', 'sanctioned', 'Russian', 'oligarchs', 'Platinum', 'Jubilee', ':', '70', 'years', 'of', 'the', 'Queen', 'in', '70', 'seconds', 'Red', 'Bull', 'found', 'guilty', 'of', 'breaking', 'Formula', '1', "'s", 'budget', 'cap', 'World', 'Triathlon', 'Championship', 'Series', ':', 'Flora', 'Duffy', 'beats', 'Georgia', 'Taylor-Brown', 'to', 'women', "'s", 'title', 'Terry', 'Hall', ':', 'Coventry', 'scooter', 'ride-out', 'pays', 'tribute', 'to', 'singer', 'Post', 'Office', 'and', 'Fujitsu', 'to', 'face', 'inquiry', 'over', 'Horizon', 'scandal', "'Pavement", 'parking', 'frightens', 'me', "'", 'UK', 'interest', 'rates', ':', 'How', 'will', 'the', 'rise', 'affect', 'you', 'and', 'how', 'high', 'could', 'it', 'go', '?', 'They', 'stayed', 'for', '

In [32]:
print(tokens_clean_list)

['refuse', 'work', 'liz', 'truss', 'brief', 'world', 'reacts', 'uk', 'political', 'turmoil', 'rationing', 'energy', 'nothing', 'new', 'offgrid', 'community', 'hunt', 'superyachts', 'sanctioned', 'russian', 'oligarchs', 'platinum', 'jubilee', '70', 'years', 'queen', '70', 'seconds', 'red', 'bull', 'found', 'guilty', 'breaking', 'formula', '1s', 'budget', 'cap', 'world', 'triathlon', 'championship', 'series', 'flora', 'duffy', 'beats', 'georgia', 'taylorbrown', 'womens', 'title', 'terry', 'hall', 'coventry', 'scooter', 'rideout', 'pays', 'tribute', 'singer', 'post', 'office', 'fujitsu', 'face', 'inquiry', 'horizon', 'scandal', 'pavement', 'parking', 'frightens', 'me', 'uk', 'interest', 'rates', 'rise', 'affect', 'high', 'could', 'go', 'stayed', 'storm', 'happens', 'now', 'six', 'nations', 'scotlands', 'best', 'since', '99', 'beat', 'best', 'ireland', 'ever', 'long', 'liz', 'truss', 'survive', 'prime', 'minister', 'platinum', 'jubilee', 'beacons', 'light', 'across', 'globe', 'celebrate', 

# POS Tagging

In [44]:
nlp = spacy.load('en_core_web_sm')
spacy_doc = nlp(' '.join(tokens_raw_list))


# 1. Faster way: Use a list comprehension to extract data
data = [{'Tokens': token.text, 'POS': token.pos_} for token in spacy_doc]

# 2. Create the DataFrame once
df_pos = pd.DataFrame(data)

# df_pos = pd.DataFrame(columns=['Tokens', 'POS'])

In [45]:
# for tokens in spacy_doc:
#     df_pos = pd.concat([
#         df_pos, pd.DataFrame.from_records([
#             { 'Tokens': tokens.text, 'POS': tokens.pos_}
#         ])
#     ], ignore_index = True)

In [46]:
df_pos_counts = df_pos.groupby(['Tokens','POS']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

In [47]:
df_pos_counts.head()

,Tokens,POS,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172


#### NER tagging


In [54]:
ner_data =  [{'Tokens': token.text, 'NER': token.label_} for token in spacy_doc.ents]
df_ner = pd.DataFrame(ner_data)
df_ner.info()
df_ner.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670 entries, 0 to 1669
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Tokens  1670 non-null   object
 1   NER     1670 non-null   object
dtypes: object(2)
memory usage: 26.2+ KB


,Tokens,NER
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [55]:
df_ner_count = df_ner.groupby(['Tokens','NER']).size().reset_index(name='counts').sort_values(by="counts", ascending=False)
df_ner_count.head()

,Tokens,NER,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19
